# 🗳️ Encuestas Presidenciales Colombia 2026 — Pipeline de Producción

**Ingesta, armoniza y analiza microdatos de las 12 encuestas oficiales registradas en el CNE.**

> Este notebook ejecuta el paquete `encuestas_lib`: lee los archivos de cada encuestadora (Atlas Intel, GAD3, Invamer, CNC), armoniza nombres de candidatos y demografías, calcula intención de voto agregada con ponderación entre encuestas, y produce un conjunto consolidado de tablas — básicas y avanzadas — listas para análisis o publicación.

Repositorio del paquete: el código vive en el ZIP `encuestas_refactor.zip` que descargaste, dentro de la carpeta `encuestas_lib/`.

---

## ✅ Antes de empezar — checklist

Cumple estos **3 pasos obligatorios** antes de correr cualquier celda:

1. **Subir el paquete y los datos a Google Drive** con la siguiente estructura:

   ```
   MyDrive/Pruebas/encuestas_presidenciales_2026/
   ├── encuestas_lib/         ← código del paquete (del ZIP)
   ├── configs/               ← archivos YAML (del ZIP)
   ├── pyproject.toml         ← (del ZIP)
   ├── requirements.txt       ← (del ZIP)
   └── data/
       └── raw/               ← AQUÍ van los archivos de las encuestas
           ├── 04. CNE-E-DG-2026-000755 - ATLAS INTEL/
           │   └── Atlas Semana E126 Raw Data 010926.xlsx
           ├── 05. CNE-E-DG-2026-001504 - GAD3/
           │   └── 505-221 RCN Enero_4.xlsx
           └── ...  (ver Step 4 para la lista completa)
   ```

2. **Runtime** estándar es suficiente (no necesita GPU). Pero conviene usar **High-RAM** si vas a procesar las 12 encuestas a la vez (44k filas, ~3GB en memoria).

3. **Conoces tu WORKSPACE**: la ruta raíz en Drive que contiene `encuestas_lib/`, `configs/` y `data/raw/`. Por defecto: `/content/drive/MyDrive/Pruebas/encuestas_presidenciales_2026`.

---

## 📋 Qué hace cada paso

| Step | Hace |
|------|------|
| 1 | Monta Google Drive |
| 2 | Configuración (ÚNICA celda que editas) |
| 3 | Instala el paquete `encuestas_lib` en modo editable |
| 4 | **Preflight check**: lista los 12 archivos esperados y reporta cuáles encontró |
| 5 | Smoke test: corre 1 sola encuesta para confirmar que todo conecta |
| 6 | Ingestión completa: lee, armoniza y consolida los 12 archivos |
| 7 | Análisis: genera ~25 tablas (básicas + avanzadas) y las exporta a Excel/JSON |
| 8 | Exploración: visualiza los resultados y corre análisis ad-hoc |

---
## ✅ Step 1 — Montar Google Drive

Esto monta tu Drive en `/content/drive/`, permitiendo que el notebook lea archivos de tu cuenta.

**Debes correr esta celda al inicio de cada sesión de Colab.** Aparecerá una ventana pidiendo autorización — sigue el link y pega el código que te dé Google.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')
print('✅ Google Drive montado en /content/drive')

---
## ✅ Step 2 — Configuración (la única celda que necesitas editar)

> 💡 **Edita SOLO esta celda.** Todas las demás reutilizan estas variables sin cambios.

### Qué hace cada parámetro

| Parámetro | Para qué sirve |
|---|---|
| `WORKSPACE` | Carpeta raíz en Drive. Debe contener `encuestas_lib/`, `configs/` y `data/raw/`. |
| `WEIGHTING_STRATEGY` | Estrategia para ponderar encuestas entre sí: `uniform`, `sample_size`, `recency_decay`, `inverse_recency_size` (default), o `manual`. Documentado en `configs/weights.yaml`. |
| `FORCE_REINGEST` | `True` reprocesa todos los archivos ignorando el checkpoint parquet. Útil cuando cambiaste el código o los datos. |
| `SKIP_MISSING_FILES` | `True` omite encuestas cuyo archivo no encontraste en Drive. Útil para probar con un subconjunto antes de tenerlas todas. |
| `RUN_SMOKE_TEST` | `True` corre el Step 5 (test rápido con 1 sola encuesta). Recomendado siempre la primera vez. |

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  ✏️  EDITAR AQUÍ — esta es la única celda que requiere cambios
# ══════════════════════════════════════════════════════════════════

# ── Carpeta raíz en tu Drive ──────────────────────────────────────
WORKSPACE = '/content/drive/MyDrive/Pruebas/encuestas_presidenciales_2026'

# ── Estrategia de ponderación entre encuestas ─────────────────────
# 'uniform'              — todas las encuestas pesan 1
# 'sample_size'          — peso proporcional al n declarado
# 'recency_decay'        — las más recientes pesan más (half-life 21 días)
# 'inverse_recency_size' — combina n × decaimiento temporal  ⭐ RECOMENDADO
# 'manual'               — pesos hardcoded en configs/weights.yaml
WEIGHTING_STRATEGY = 'inverse_recency_size'

# ── Opciones de ejecución ─────────────────────────────────────────
FORCE_REINGEST = False        # True = ignora checkpoint y reprocesa todo
SKIP_MISSING_FILES = True     # True = omite encuestas sin archivo (útil al empezar)
RUN_SMOKE_TEST = True         # True = corre Step 5 antes de la ingesta completa

# ── Nombres de los archivos de salida ─────────────────────────────
OUTPUT_EXCEL = 'analisis_consolidado.xlsx'
OUTPUT_JSON  = 'analisis_consolidado.json'

print(f'📁 Workspace: {WORKSPACE}')
print(f'⚖️  Estrategia de pesos: {WEIGHTING_STRATEGY}')
print(f'🔁 Forzar reingesta: {FORCE_REINGEST}')
print(f'⏭️  Omitir faltantes: {SKIP_MISSING_FILES}')

---
## ✅ Step 3 — Instalar el paquete `encuestas_lib`

Instala el paquete en **modo editable** (`pip install -e .`) desde tu Drive. Modo editable significa que si modificas el código fuente en `encuestas_lib/`, los cambios se reflejan al re-ejecutar las celdas sin reinstalar.

> ⚠️ **Solo la PRIMERA vez** que corres esto en una sesión nueva de Colab: después de instalar, ve a `Entorno de ejecución → Reiniciar entorno de ejecución` y vuelve a correr desde el Step 1. Es un requisito de Colab — los paquetes recién instalados no son visibles hasta que reinicies el kernel.

In [ ]:
%%time
import subprocess, sys
from pathlib import Path

PKG_ROOT = Path(WORKSPACE)

if not (PKG_ROOT / 'pyproject.toml').exists():
    raise FileNotFoundError(
        f'❌ No se encontró pyproject.toml en {PKG_ROOT}.\n'
        f'   Verifica que descomprimiste encuestas_refactor.zip en esa carpeta.'
    )

print(f'📦 Instalando desde {PKG_ROOT} (modo editable)…')
result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-e', str(PKG_ROOT), '--quiet'],
    capture_output=True, text=True,
)

if result.returncode == 0:
    print('✅ encuestas_lib instalado en modo editable')
    print('⚠️  Si es la PRIMERA instalación: Entorno → Reiniciar entorno y volver al Step 1')
else:
    print('❌ Falló la instalación:')
    print(result.stderr[-1500:])

---
## ✅ Step 4 — Preflight check: ¿qué archivos están en Drive?

Esta celda **NO procesa nada todavía**. Solo audita: lee el registro de encuestas en `configs/surveys.yaml` y verifica cuáles archivos están en tu Drive.

### 📂 Lista exacta de archivos que el pipeline espera en `data/raw/`

Subcarpetas con sus nombres tal cual el CNE las publica:

| # | Encuestadora | Fecha | Archivo |
|---|---|---|---|
| 1 | Atlas Intel | 2026-01-09 | `04. CNE-E-DG-2026-000755 - ATLAS INTEL/Atlas Semana E126 Raw Data 010926.xlsx` |
| 2 | Atlas Intel | 2026-02-05 | `13. CNE-E-DG-2026-004955 - ATLAS INTEL/Atlas Semana E226 Raw Data 020526_1 (1).csv` |
| 3 | Atlas Intel | 2026-02-28 | `27. CNE-E-DG-2026-008698 - ATLAS INTEL/Base de Dados Atlas Semana 022826_2.xlsx` |
| 4 | Atlas Intel | 2026-03-12 | `29. CNE-E-DG-2026-011140 - ATLAS INTEL/Base de Dados Atlas Semana 031226_1.xlsx` |
| 5 | Atlas Intel | 2026-04-10 | `36. CNE-E-DG-2026-014018 - ATLAS INTEL/Base de Dados Atlas Semana 041026_1.xlsx` |
| 6 | GAD3 | 2026-01-13 | `05. CNE-E-DG-2026-001504 - GAD3/505-221 RCN Enero_4.xlsx` |
| 7 | GAD3 | 2026-02-16 | `23. CNE-E-DG-2026-008418 - GAD3/Anexo IV. B. Microdatos estudio (SPSS).sav` |
| 8 | GAD3 | 2026-03-16 | `31. CNE-E-DG-2026-011911 - GAD3/Anexo IV.A.Microdatos estudio (Excel).xlsx` |
| 9 | Invamer | 2026-02-25 | `19. CNE-E-DG-2026-008198 - INVAMER/3. Data (Regsitros primarios).xlsx` |
| 10 | Invamer | 2026-04-25 | `38. CNE-E-DG-2026-015879 - INVAMER/3. Data (Regsitros primarios).xlsx` |
| 11 | CNC | 2026-02-20 | `25. CNE-E-DG-2026-009065 - CENTRO NACIONAL DE CONSULTORÍA/RevCambioFebrero/data/CC892901_BASE_REVISTA_CAMBIO.sav` |
| 12 | CNC | 2026-03-16 | `34. CNE-E-DG-2026-012431 - CENTRO NACIONAL DE CONSULTORIA/Base de datos/CC893201_POLITICA_REVISTA_CAMBIO_TEXTOS.xlsx` |

> 💡 **Atención a los detalles:**
> - Los nombres y prefijos numéricos (`04.`, `13.`, etc.) deben coincidir **exactamente**. Drive y Linux son case-sensitive.
> - Las dos Invamer (`19.` y `38.`) tienen el archivo con el **mismo nombre** dentro — están en subcarpetas distintas.
> - La CNC del 20 de feb tiene un nivel extra: `RevCambioFebrero/data/...`
> - Para agregar una encuesta nueva: edita `configs/surveys.yaml` y agrega un bloque al final. No requiere tocar código si la encuestadora ya tiene reader.

In [ ]:
from encuestas_lib.config import Config, WeightingConfig
from encuestas_lib.pipeline import IngestPipeline
from dataclasses import replace

# Cargar configuración desde configs/ (YAMLs)
config = Config.from_yaml(
    configs_dir=f'{WORKSPACE}/configs',
    root=WORKSPACE,  # data/raw vivirá en {WORKSPACE}/data/raw
)

# Forzar la estrategia de pesos elegida en la celda de configuración
from encuestas_lib.config import WeightingConfig
config = replace(config, weighting=replace(config.weighting, active_strategy=WEIGHTING_STRATEGY))

print(f'✅ Config cargada: {len(config.surveys)} encuestas registradas')
print(f'   Estrategia activa: {config.weighting.active_strategy}')
print(f'   Datos crudos en:  {config.paths.raw_dir}')
print(f'   Outputs en:       {config.paths.outputs_dir}')

# Reporte preflight
ingest = IngestPipeline(config)
report = ingest.preflight()
report

---
## ✅ Step 5 — Smoke test (1 sola encuesta)

**No saltes este paso.** Lee la primera encuesta disponible en menos de 30 segundos y confirma que:

- El reader correspondiente puede leer el archivo (formato, encoding correctos)
- La armonización de nombres de candidatos funciona
- El schema canónico de salida está completo

Si el smoke test falla, **el problema lo encontrarás en 30 segundos** en vez de gastar 5 minutos en la ingesta completa.

> Pon `RUN_SMOKE_TEST = False` en el Step 2 para saltarlo (no recomendado la primera vez).

In [ ]:
if RUN_SMOKE_TEST:
    from encuestas_lib.readers import get_reader_class
    from encuestas_lib.harmonization import build_harmonizer
    import time

    # Encuestas disponibles según el preflight
    disponibles = [s for s in config.surveys if s.path.exists()]
    if not disponibles:
        raise FileNotFoundError(
            '⚠️  Ninguna encuesta encontrada. Sube al menos un archivo a data/raw/ '
            'siguiendo la tabla del Step 4.'
        )

    primer = disponibles[0]
    print(f'🧪 Smoke test con: {primer.id} ({primer.path.name})')

    harmonizer = build_harmonizer(config.candidates_raw, config.special_categories_raw)
    ReaderCls = get_reader_class(primer.reader)

    t0 = time.time()
    df_smoke = ReaderCls(primer, harmonizer).read()
    elapsed = time.time() - t0

    print(f'\n✅ Smoke test PASÓ en {elapsed:.1f}s')
    print(f'   Filas leídas:          {len(df_smoke):,}')
    print(f'   Columnas canónicas:    {len(df_smoke.columns)}')
    print(f'   Candidatos detectados: {df_smoke["primera_vuelta"].nunique()}')
    print(f'\nTop 5 candidatos en esta encuesta:')
    print(df_smoke["primera_vuelta"].value_counts().head().to_string())
else:
    print('⏭️  Smoke test omitido (RUN_SMOKE_TEST=False)')

---
## ✅ Step 6 — Ingestión completa

Lee todos los archivos disponibles, los armoniza al schema canónico (mismas columnas y nombres de candidato en todas las encuestas), y los concatena en un único DataFrame.

El resultado se cachea en `data/processed/encuestas_concatenadas.parquet`. Las siguientes ejecuciones leen el parquet directamente (segundos en vez de minutos), salvo que pongas `FORCE_REINGEST=True`.

**Tiempo esperado**: ~2-4 minutos para las 12 encuestas en Colab estándar.

In [ ]:
%%time
df = ingest.run(forzar=FORCE_REINGEST, skip_missing=SKIP_MISSING_FILES)

print(f'\n📊 DataFrame consolidado:')
print(f'   Filas:        {len(df):,}')
print(f'   Columnas:     {len(df.columns)}')
print(f'   Encuestas:    {df[["encuestadora", "fecha"]].drop_duplicates().shape[0]}')
print(f'   Rango fechas: {df["fecha"].min()} a {df["fecha"].max()}')

In [ ]:
# Filas por encuestadora
df.groupby('encuestadora').size().sort_values(ascending=False).to_frame('n_filas')

---
## ✅ Step 7 — Análisis y exportación

Corre el `AnalysisPipeline`: genera **~25 tablas** (entre básicas y avanzadas) y las exporta como:

- **Excel multi-hoja** en `data/outputs/analisis_consolidado.xlsx` (una hoja por tabla)
- **JSON con metadatos** en `data/outputs/analisis_consolidado.json` (timestamp, hash, todas las tablas)

Al final imprime un reporte de validación forense: cuántas tablas cierran a 100% (auditoría automática).

### Tablas generadas

**Básicas (reproducen el repo original):**
- `primera_vuelta_total` — intención de voto agregada
- `voto_por_region`, `voto_por_edad`, `voto_por_genero`
- `aprobacion_vs_voto`, `voto_vs_aprobacion`
- `sesgo_genero`, `sesgo_edad`, `sesgo_region` — house effects
- `indecisos_total`, `indecisos_region`, `indecisos_edad_grupo`, `indecisos_sexo`

**Avanzadas (NUEVAS):**
- `trend_primera_vuelta` — serie temporal con suavizado por ventana móvil
- `coalicion_aprobacion` — voto × aprobación Petro
- `volatilidad_encuestadora` — desviación intra-pollster
- `indecisos_perfil` — tasa de indecisión por dimensión
- `moe_*` — margen de error con n efectivo de Kish (para top-3 candidatos)
- `transfer_sv_*` — transferencia PV → SV por matchup
- `techo_potencial_*` — espacio de crecimiento PV → SV
- `_validacion_cierres` — auditoría forense de cierres a 100

In [ ]:
%%time
from encuestas_lib.pipeline import AnalysisPipeline

tablas = AnalysisPipeline(config).run(
    df,
    excel_name=OUTPUT_EXCEL,
    json_name=OUTPUT_JSON,
    validar=True,
)

print(f'\n📋 Total de tablas generadas: {len(tablas)}')

---
## ✅ Step 8 — Exploración de resultados

Las tablas viven en el dict `tablas` (en memoria) y también en disco (Excel + JSON).

### 8.1 — Intención de voto agregada en primera vuelta

In [ ]:
tablas['primera_vuelta_total'].sort_values('valor', ascending=False)

### 8.2 — Tendencia temporal (top-5 candidatos)

Esta tabla está en formato long: `[fecha, primera_vuelta, valor_punto, valor_suavizado]`. Cada candidato tiene una serie temporal con dos columnas: el punto observado por encuesta y la media móvil ponderada por ventana de 14 días.

In [ ]:
trend = tablas['trend_primera_vuelta']
top5 = (tablas['primera_vuelta_total']
        .nlargest(5, 'valor')['primera_vuelta'].tolist())
trend[trend['primera_vuelta'].isin(top5)].head(30)

### 8.3 — Validación forense de cierres a 100%

Cada tabla que **debe** sumar 100% (intención de voto, demografías, etc.) es auditada automáticamente. Tolerancia: 0.5 puntos porcentuales por defecto.

In [ ]:
tablas['_validacion_cierres']

### 8.4 — Margen de error efectivo (IC95% con n de Kish)

Para cada encuesta, calcula el margen de error a 95% **corrigiendo el n nominal por la dispersión de los factores de expansión**. Si los factores son muy heterogéneos, el n efectivo es bastante menor que el n declarado — y eso amplía el IC.

In [ ]:
# Para el candidato #1 (líder)
lider_key = sorted(
    [k for k in tablas if k.startswith('moe_')]
)[0]
tablas[lider_key]

### 8.5 — House effects (sesgo de cada encuestadora vs el promedio del resto)

Si una encuestadora reporta sistemáticamente más mujeres que el resto, esto lo muestra. Útil para auditar la metodología.

In [ ]:
tablas['sesgo_region'].head(20)

### 8.6 — Volatilidad intra-encuestadora

Desviación estándar entre mediciones de la misma encuestadora. Volatilidad alta puede ser ruido metodológico, o captura legítima de movimiento — no se puede saber sin más contexto.

In [ ]:
tablas['volatilidad_encuestadora'].head(20)

---
## 🛠️ Step 9 — Análisis ad-hoc (EXTRAS)

Las funciones del módulo `encuestas_lib.analysis` son puras: reciben el DataFrame y devuelven un DataFrame. Útiles para preguntas que no están en el pipeline estándar.

### 9.1 — Transferencia de voto PV → SV para una matchup específica

In [ ]:
from encuestas_lib.analysis import transferencia_pv_sv, resolve_weights, sv_columns

pesos = resolve_weights(config.weighting, config.surveys)
sv_disponibles = sv_columns(df)
print('Matchups SV disponibles:', sv_disponibles)

# Ejemplo: transferencia para la primera matchup
if sv_disponibles:
    transferencia_pv_sv(df, sv_disponibles[0], pesos)

### 9.2 — Perfil de indecisos por dimensión

In [ ]:
from encuestas_lib.analysis import indecisos_perfil
from encuestas_lib.harmonization import build_harmonizer

harm = build_harmonizer(config.candidates_raw, config.special_categories_raw)
indecisos_perfil(df, harm.vigentes())

### 9.3 — Margen de error para cualquier candidato

Cambia el nombre canónico para inspeccionar el IC de otro candidato.

In [ ]:
from encuestas_lib.analysis import margen_error_efectivo

margen_error_efectivo(df, 'Iván Cepeda')

---
## ✅ Step 10 — Verificación de outputs

Confirma que los archivos finales se generaron correctamente en Drive.

In [ ]:
from pathlib import Path

out_dir = Path(WORKSPACE) / 'data' / 'outputs'
for f in sorted(out_dir.iterdir()):
    size_mb = f.stat().st_size / 1e6
    print(f'   {f.name:<40} {size_mb:>6.2f} MB')

print(f'\n📂 Ubicación: {out_dir}')
print('   Estos archivos están en tu Drive y puedes descargarlos o compartirlos.')

---

## 🎉 Pipeline terminado

Si llegaste aquí sin errores, tienes:

- ✅ 12 encuestas armonizadas en un único DataFrame (`df`)
- ✅ ~25 tablas analíticas en `tablas`, exportadas a Excel y JSON en Drive
- ✅ Auditoría forense que confirma que las tablas cierran a 100%

### Próximos pasos

1. **Cuando agregues una encuesta nueva**: edita `configs/surveys.yaml`, sube el archivo a `data/raw/`, vuelve a correr el notebook. No requiere tocar código si el formato ya está soportado.
2. **Si llega una encuestadora nueva** (Guarumo, Datexco): debes crear un Reader nuevo en `encuestas_lib/readers/` heredando de `BaseReader`. Ver `README.md` § 'Agregar una nueva encuesta — Caso 2'.
3. **Para publicar resultados**: documenta la estrategia de pesos que usaste (está en `configs/weights.yaml`). Antes de publicar, justifica por escrito por qué elegiste `inverse_recency_size` (o la que sea). Si no puedes defenderlo, alguien te lo va a desmontar.